# CSAI415 — Deliverable 2: Retrieval Stack Demo

**Team:** Baraa · Yousef · Khalid · Salim · Faisal

This notebook demonstrates the full D2 pipeline end-to-end:
1. Stack health check
2. Architecture dataflow diagram
3. Hybrid search (BM25 + Dense) with top-k citations
4. Neo4j graph — 5 example Cypher queries
5. Evaluation metrics (Recall@k, latency)

In [1]:
import requests

API_URL = "http://localhost:8000"

stats = requests.get(f"{API_URL}/stats").json()
print("Stack health:")
for k, v in stats.items():
    print(f"  {k}: {v}")

Stack health:
  mongo_chunks: 2640
  mongo_docs: 100
  mongo_feedback: 0
  qdrant_vectors: 2640


## Architecture — Dataflow Diagram

```
  ┌─────────────┐
  │  PDF Files  │  (100 arXiv papers, RAG/IR domain)
  └──────┬──────┘
         │
         ▼
  ┌─────────────┐     text + chunks      ┌──────────────────┐
  │  ingest.py  │ ─────────────────────► │  MongoDB (d2.*)  │
  │  pypdf      │                        │  chunks  2 640   │
  └──────┬──────┘                        │  docs      100   │
         │                              └──────────────────┘
         │  BGE-small embeddings
         ▼
  ┌──────────────────┐
  │  vector_store.py │ ─── 384-dim vectors ──► Qdrant  2 640 pts
  └──────────────────┘
         │
         │  arXiv metadata (enrich.py)
         ▼
  ┌──────────────────┐
  │  graph_builder.py│ ──► Neo4j: 100 Papers · 510 Authors · 372 Topics
  └──────────────────┘

  ─────────────── Query Time ───────────────

  User Query
       │
       ├──► BM25 (rank-bm25, MongoDB chunks) ──┐
       │                                        ├──► RRF fusion ──► Top-k results
       └──► Dense (BGE + Qdrant ANN) ───────────┘     + doc-level aggregation
```

## Hybrid Search — Top-k Examples with Citations

In [2]:
DEMO_QUERIES = [
    "retrieval augmented generation survey",
    "knowledge graph question answering",
    "dense passage retrieval transformers",
    "BM25 sparse retrieval benchmark",
    "LLM evaluation reasoning tasks",
]

for query in DEMO_QUERIES:
    results = requests.post(
        f"{API_URL}/search",
        json={"query": query, "top_k": 3}
    ).json()
    print(f'Query: "{query}"')
    for i, r in enumerate(results, 1):
        print(f"  [{i}] {r['title']}")
        print(f"       doc_id={r['doc_id']}  page={r['page']}  score={r['score']:.4f}")
    print()

Query: "retrieval augmented generation survey"
  [1] Seeking Information with RAG-Assistants: Does Model Size Matter in Human-AI Collaborations?
       doc_id=2605.00964v1  page=13  score=0.0306
  [2] Verbal-R3: Verbal Reranker as the Missing Bridge between Retrieval and Reasoning
       doc_id=2605.01399v1  page=10  score=0.0280
  [3] AgenticRAG: Agentic Retrieval for Enterprise Knowledge Bases
       doc_id=2605.05538v1  page=8  score=0.0279

Query: "knowledge graph question answering"
  [1] FollowTable: A Benchmark for Instruction-Following Table Retrieval
       doc_id=2605.00400v1  page=11  score=0.0303
  [2] Text-Graph Synergy: A Bidirectional Verification and Completion Framework for RAG
       doc_id=2605.05643v1  page=5  score=0.0266
  [3] Hierarchical Abstract Tree for Cross-Document Retrieval-Augmented Generation
       doc_id=2605.00529v1  page=13  score=0.0265

Query: "dense passage retrieval transformers"
  [1] A Hybrid Retrieval and Reranking Framework for Evidence-Groun

## Neo4j Graph — 5 Example Cypher Queries

In [3]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv("../.env.local", override=True)

driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI", "bolt://localhost:7687"),
    auth=(os.getenv("NEO4J_USER", "neo4j"), os.getenv("NEO4J_PASS", "changeme123"))
)

def run(cypher):
    with driver.session() as s:
        return s.run(cypher).data()

In [4]:
# Query 1 — Papers per year
print("Query 1: Papers per year")
for row in run("MATCH (p:Paper) RETURN p.year AS year, count(p) AS papers ORDER BY year DESC"):
    print(f"  {row}")

Query 1: Papers per year
  {'year': 2026, 'papers': 100}


In [5]:
# Query 2 — Most prolific authors
print("Query 2: Most prolific authors (top 10)")
cypher = """
MATCH (p:Paper)-[:AUTHORED_BY]->(a:Author)
RETURN a.name AS author, count(p) AS papers
ORDER BY papers DESC LIMIT 10
"""
for row in run(cypher):
    print(f"  {row['author']}: {row['papers']} paper(s)")

Query 2: Most prolific authors (top 10)
  Wataru Uegami: 2 paper(s)
  Xi Wang: 2 paper(s)
  Tingyu Song: 2 paper(s)
  Saghir Alfasly: 2 paper(s)
  Wei Ye: 2 paper(s)
  Di Liang: 2 paper(s)
  Yilun Zhao: 2 paper(s)
  Siyue Zhang: 2 paper(s)
  H. R. Tizhoosh: 2 paper(s)
  Ziqiang Cui: 2 paper(s)


In [6]:
# Query 3 — Most common topics
print("Query 3: Most common topics (top 10)")
cypher = """
MATCH (p:Paper)-[:HAS_TOPIC]->(t:Topic)
RETURN t.name AS topic, count(p) AS papers
ORDER BY papers DESC LIMIT 10
"""
for row in run(cypher):
    print(f"  {row['topic']}: {row['papers']} papers")

Query 3: Most common topics (top 10)
  generation: 16 papers
  augmented: 14 papers
  aware: 13 papers
  recommendation: 13 papers
  agent: 10 papers
  search: 9 papers
  generative: 6 papers
  reasoning: 6 papers
  information: 5 papers
  recommender: 5 papers


In [7]:
# Query 4 — Papers sharing topic 'agent'
print("Query 4: Papers on topic 'agent'")
cypher = """
MATCH (p:Paper)-[:HAS_TOPIC]->(t:Topic {name: 'agent'})
RETURN p.doc_id AS doc_id, p.title AS title
LIMIT 5
"""
for row in run(cypher):
    print(f"  {row['doc_id']}: {row['title'][:70]}")

Query 4: Papers on topic 'agent'
  2605.04003v1: Physics-Grounded Multi-Agent Architecture for Traceable, Risk-Aware Hu
  2605.06647v1: Superintelligent Retrieval Agent: The Next Frontier of Information Ret
  2605.05991v1: A Case-Driven Multi-Agent Framework for E-Commerce Search Relevance
  2605.05287v1: Securing the Agent: Vendor-Neutral, Multitenant Enterprise Retrieval a
  2605.05250v1: Decision-aware User Simulation Agent for Evaluating Conversational Rec


In [8]:
# Query 5 — Co-author pairs
print("Query 5: Co-author pairs (top 10)")
cypher = """
MATCH (a1:Author)<-[:AUTHORED_BY]-(p:Paper)-[:AUTHORED_BY]->(a2:Author)
WHERE a1.name < a2.name
RETURN a1.name AS author1, a2.name AS author2, count(p) AS shared_papers
ORDER BY shared_papers DESC LIMIT 10
"""
for row in run(cypher):
    print(f"  {row['author1']} & {row['author2']}: {row['shared_papers']} shared")

Query 5: Co-author pairs (top 10)


  Saghir Alfasly & Wataru Uegami: 2 shared
  Wei Ye & Xi Wang: 2 shared
  Siyue Zhang & Tingyu Song: 2 shared
  H. R. Tizhoosh & Wataru Uegami: 2 shared
  Di Liang & Wei Ye: 2 shared
  Peiyang Liu & Wei Ye: 2 shared
  Siyue Zhang & Yilun Zhao: 2 shared
  Tingyu Song & Yilun Zhao: 2 shared
  H. R. Tizhoosh & Saghir Alfasly: 2 shared
  Di Liang & Xi Wang: 2 shared


## Neo4j Graph — Visualization

Full graph: **100 Papers** (red) · **510 Authors** (teal) · **523 AUTHORED_BY relationships**

**Overview — all 100 paper clusters:**

![full graph](../Visualizations/neo4j_full.jpeg)

**Zoomed in — co-author relationships per paper:**

![zoomed graph](../Visualizations/neo4j_zoom.jpeg)

> Query: `MATCH (p:Paper)-[r:AUTHORED_BY]->(a:Author) RETURN p, r, a LIMIT 600`

## Evaluation Metrics

In [9]:
# Reproduce metrics live
import sys
sys.path.insert(0, "..")
from src.evaluate import evaluate
evaluate()

Evaluating 10 queries against http://localhost:8000/search

  HIT   Q: knowledge graph traversal document injection        lat=211ms  top1=2604.27820v1
  HIT   Q: retrieval augmented generation evidence explicit    lat=90ms  top1=2605.01664v1
  MISS  Q: unified toolkit benchmark evaluating information r  lat=97ms  top1=2605.00631v1
  HIT   Q: text to sql accuracy recurring questions reliable   lat=96ms  top1=2604.28028v1
  HIT   Q: multivector retrieval token aware clustering hiera  lat=77ms  top1=2604.28142v1
  HIT   Q: LLM biases manipulate AI search overview            lat=74ms  top1=2605.00012v1
  HIT   Q: reasoning intensive retrieval survey progress chal  lat=74ms  top1=2605.00063v1
  HIT   Q: structured attribution small language models table  lat=84ms  top1=2605.00199v2
  HIT   Q: retrieval augmented reasoning chartered accountanc  lat=56ms  top1=2605.00257v1
  HIT   Q: structure aware chunking tabular data retrieval ge  lat=69ms  top1=2605.00318v1

----------------------------

{'recall': {1: 0.8, 3: 0.9, 5: 0.9},
 'mean_latency_ms': 92.7,
 'std_latency_ms': 43.3}


**Notes:**
- BM25 + BGE-small-en-v1.5 dense vectors fused with Reciprocal Rank Fusion (k=60)
- Doc-level score aggregation prevents one paper's chunks from dominating results
- HybridSearch singleton loaded at startup — no per-request BM25 rebuild